# 02 - Model A: Standard CNN

Build, train, and evaluate **Model A** (a standard CNN: interleaved `Conv2D` + `MaxPooling2D` layers) on the [Intel Image Classification](https://www.kaggle.com/datasets/puneet6060/intel-image-classification) dataset - 6 natural scene classes (buildings, forest, glacier, mountain, sea, street), resized to 64x64 per the assignment spec.

**Note:** `01_data_preparation.ipynb` is not finalized yet, so Section 4 below loads and splits the raw dataset itself (own 70/15/15 stratified split, fixed seed). Once notebook 01 produces canonical split metadata under `data/splits/`, replace Section 4 to load from there instead, so every model notebook (02, 03, 06, 07) trains/evaluates on identical splits.

In [ ]:
# --- Environment setup (Google Colab + shared Google Drive) ---
import os

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    print("Not running in Colab - skipping Drive mount.")

# Clone the repo (if not already inside it) so relative paths like results/...
# resolve correctly, and cd into it.
REPO_DIR = "EN3150-A03-Resource-Constrained-CNN"
REPO_URL = "https://github.com/ChamoDa01/EN3150-A03-Resource-Constrained-CNN.git"

if IN_COLAB and os.path.basename(os.getcwd()) != REPO_DIR:
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL}
    %cd {REPO_DIR}


## 1. Config

In [ ]:
# --- Config for this notebook (defined here, not in a shared file) ---
CONFIG = {
    "dataset_name": "Intel Image Classification (Kaggle)",
    "dataset_source": "https://www.kaggle.com/datasets/puneet6060/intel-image-classification",
    "class_names": ["buildings", "forest", "glacier", "mountain", "sea", "street"],
    "num_classes": 6,

    "image_size": 64,          # max resolution per assignment spec (<= 64x64)
    "channels": 3,

    "train_ratio": 0.70,
    "val_ratio": 0.15,
    "test_ratio": 0.15,
    "random_seed": 42,

    "batch_size": 32,
    "epochs": 20,               # assignment requires >= 20 epochs
    "learning_rate": 0.001,
    "optimizer": "adam",        # compared against SGD / SGD+momentum in 04_optimizer_comparison.ipynb

    # Dataset location on the shared Google Drive (uploaded once by the team
    # from the Kaggle zip). Expected structure inside this folder:
    #   seg_train/seg_train/<class_name>/*.jpg
    #   seg_test/seg_test/<class_name>/*.jpg
    "drive_dataset_dir": "/content/drive/MyDrive/EN3150_A03_Shared/dataset/intel-image-classification",

    # Optional backup location for the trained checkpoint (models/ is gitignored locally).
    "drive_checkpoint_dir": "/content/drive/MyDrive/EN3150_A03_Shared/models/model_a",
}


## 2. Imports

In [ ]:
import json
import random
import shutil
import time
import collections
import pathlib

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    classification_report,
)


## 3. Seed

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


set_seed(CONFIG["random_seed"])


## 4. Load Prepared Data

In [ ]:
def list_image_paths_and_labels(dataset_dir: str, class_names: list[str]):
    """Walk the Kaggle Intel dataset's seg_train/seg_test folders and collect
    (filepath, label_index) pairs. Both folders are combined here because the
    assignment requires us to create our own 70/15/15 split rather than use
    Kaggle's original train/test split."""
    dataset_dir = pathlib.Path(dataset_dir)
    subfolders = ["seg_train/seg_train", "seg_test/seg_test"]
    filepaths, labels = [], []
    for sub in subfolders:
        base = dataset_dir / sub
        if not base.is_dir():
            continue
        for class_idx, class_name in enumerate(class_names):
            class_dir = base / class_name
            if not class_dir.is_dir():
                continue
            for img_path in class_dir.glob("*.jpg"):
                filepaths.append(str(img_path))
                labels.append(class_idx)
    return filepaths, labels


filepaths, labels = list_image_paths_and_labels(CONFIG["drive_dataset_dir"], CONFIG["class_names"])
num_classes = CONFIG["num_classes"]
print(f"Found {len(filepaths)} images across {num_classes} classes.")


In [ ]:
# 70/15/15 stratified split: first split off train, then split the remainder
# evenly (in the configured ratio) into val/test.
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    filepaths, labels,
    train_size=CONFIG["train_ratio"],
    stratify=labels,
    random_state=CONFIG["random_seed"],
)

val_ratio_of_temp = CONFIG["val_ratio"] / (CONFIG["val_ratio"] + CONFIG["test_ratio"])
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels,
    train_size=val_ratio_of_temp,
    stratify=temp_labels,
    random_state=CONFIG["random_seed"],
)

print(f"Train: {len(train_paths)}  Val: {len(val_paths)}  Test: {len(test_paths)}")

# Sanity check: class balance should be roughly preserved across splits.
for split_name, split_labels in [("train", train_labels), ("val", val_labels), ("test", test_labels)]:
    counts = collections.Counter(split_labels)
    print(split_name, {CONFIG["class_names"][k]: counts[k] for k in sorted(counts)})


In [ ]:
def load_and_preprocess(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=CONFIG["channels"])
    image = tf.image.resize(image, [CONFIG["image_size"], CONFIG["image_size"]])
    image = image / 255.0
    return image, label


def make_dataset(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=CONFIG["random_seed"])
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(CONFIG["batch_size"]).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(train_paths, train_labels, shuffle=True)
val_ds = make_dataset(val_paths, val_labels, shuffle=False)
test_ds = make_dataset(test_paths, test_labels, shuffle=False)


## 5. Build Model A

In [ ]:
def build_model_a(input_shape=(64, 64, 3), num_classes=6):
    """Model A: standard CNN - interleaved Conv2D and MaxPooling2D layers,
    ending in a small dense classification head (see Figure 1 in the brief)."""
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),

        tf.keras.layers.Conv2D(32, kernel_size=3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2),

        tf.keras.layers.Conv2D(64, kernel_size=3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2),

        tf.keras.layers.Conv2D(128, kernel_size=3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ], name="model_a_standard_cnn")
    return model


model = build_model_a(
    input_shape=(CONFIG["image_size"], CONFIG["image_size"], CONFIG["channels"]),
    num_classes=CONFIG["num_classes"],
)


## 6. Model Summary

In [ ]:
model.summary()


## 7. Parameter Count

In [ ]:
total_params = model.count_params()
print(f"Total trainable parameters: {total_params:,}")


## 8. Activation Function Justification

**ReLU** (`max(0, x)`) is used in every hidden `Conv2D`/`Dense` layer, and **softmax** only at the output layer (required to produce a multi-class probability distribution).

- ReLU is a single comparison-and-clamp operation - no exponential - so it is far cheaper to evaluate on constrained/edge hardware than `sigmoid`/`tanh`, both of which require an exponential per activation, across every pixel of every feature map.
- ReLU also avoids the vanishing-gradient problem `sigmoid`/`tanh` suffer from as networks get deeper, which matters here since Model A stacks three convolutional stages.
- Softmax's exponential cost is negligible in comparison: it is evaluated once per forward pass over only `num_classes` (6) values, not per feature-map pixel like the hidden-layer activations.

## 9. Training (>= 20 epochs)

In [ ]:
class EpochTimer(tf.keras.callbacks.Callback):
    """Records wall-clock training time for each epoch, in seconds."""

    def __init__(self):
        super().__init__()
        self.epoch_times = []
        self._epoch_start = None

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        self.epoch_times.append(time.time() - self._epoch_start)


checkpoint_dir = "models/checkpoints/model_a"
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, "best.keras")

optimizer = tf.keras.optimizers.Adam(learning_rate=CONFIG["learning_rate"])
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

epoch_timer = EpochTimer()
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    checkpoint_path, monitor="val_loss", save_best_only=True,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG["epochs"],
    callbacks=[checkpoint_cb, epoch_timer],
)

avg_epoch_time_sec = float(np.mean(epoch_timer.epoch_times))
print(f"Average training time per epoch: {avg_epoch_time_sec:.2f} s")


In [ ]:
# Optional: back up the trained checkpoint to the shared Drive so teammates
# can reuse it without retraining.
try:
    if IN_COLAB:
        drive_checkpoint_dir = CONFIG["drive_checkpoint_dir"]
        os.makedirs(drive_checkpoint_dir, exist_ok=True)
        shutil.copy(checkpoint_path, os.path.join(drive_checkpoint_dir, "best.keras"))
        print(f"Backed up checkpoint to {drive_checkpoint_dir}")
except Exception as exc:
    print(f"Skipped Drive checkpoint backup: {exc}")


## 10. Training / Validation Loss Curves

In [ ]:
os.makedirs("results/figures/model_a", exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Model A - Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Model A - Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

fig.tight_layout()
fig.savefig("results/figures/model_a/loss_curve.png", dpi=150)
plt.show()


## 11. Test Set Evaluation

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

test_accuracy = accuracy_score(y_true, y_pred)
test_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
test_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
cm = confusion_matrix(y_true, y_pred)

print(f"Test accuracy:            {test_accuracy:.4f}")
print(f"Test precision (macro):   {test_precision:.4f}")
print(f"Test recall (macro):      {test_recall:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=CONFIG["class_names"]))


## 12. Confusion Matrix

In [ ]:
os.makedirs("results/confusion_matrices", exist_ok=True)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(CONFIG["num_classes"]))
ax.set_yticks(range(CONFIG["num_classes"]))
ax.set_xticklabels(CONFIG["class_names"], rotation=45, ha="right")
ax.set_yticklabels(CONFIG["class_names"])
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Model A - Confusion Matrix")

for i in range(CONFIG["num_classes"]):
    for j in range(CONFIG["num_classes"]):
        ax.text(
            j, i, cm[i, j], ha="center", va="center",
            color="white" if cm[i, j] > cm.max() / 2 else "black",
        )

fig.colorbar(im)
fig.tight_layout()
fig.savefig("results/confusion_matrices/model_a.png", dpi=150)
plt.show()


## 13. Model Size & Training Time

In [ ]:
model_size_kb = os.path.getsize(checkpoint_path) / 1024
print(f"Model A checkpoint size: {model_size_kb:.1f} KB")
print(f"Average training time per epoch: {avg_epoch_time_sec:.2f} s")


## 14. Save Results

In [ ]:
os.makedirs("results/metrics", exist_ok=True)

metrics = {
    "model_name": "model_a",
    "dataset_name": CONFIG["dataset_name"],
    "num_classes": CONFIG["num_classes"],
    "class_names": CONFIG["class_names"],
    "total_params": int(total_params),
    "model_size_kb": model_size_kb,
    "epochs": CONFIG["epochs"],
    "avg_epoch_time_sec": avg_epoch_time_sec,
    "test_accuracy": float(test_accuracy),
    "test_precision_macro": float(test_precision),
    "test_recall_macro": float(test_recall),
    "confusion_matrix": cm.tolist(),
    "history": {k: [float(v) for v in vals] for k, vals in history.history.items()},
}

with open("results/metrics/model_a.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved results/metrics/model_a.json")


## 15. Conclusion

TODO (fill in after running this notebook in Colab): summarize Model A's total parameter count, test accuracy/precision/recall, and any qualitative observations from the confusion matrix (e.g. which scene classes are most often confused, such as `glacier` vs. `mountain`, or `sea` vs. `buildings` for coastal cityscapes). Once Model B (`03_model_B_lightweight_cnn.ipynb`) is done, `05_custom_models_evaluation.ipynb` will compare both on parameter count, model size, training time/epoch, and accuracy.